# CPLEX Sanity Check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cplex
from docplex.mp.model import Model
import torch
import torch.optim as optim
from itertools import permutations, combinations

### Define the synthetic network

In [ ]:
nodes = 6
M = 2
alpha = np.array([[1,1,0,1,0,0], [1,1,0,0,1,0], [0,0,1,1,0,1], [1,0,1,1,0,0], [0,1,0,0,1,1], [0,0,1,0,1,1]])

theta = np.array([0.1, 0.15, 0.15, 0.85, 0.8, 0.9])
omega = np.array([[0.8, .2, .3, .5 ,.2, .3], [.3, .2, 0.5, 0.5, .6, .2], [0.9, .2, .3, 0.8, .6, .7], [0.9, .2, .4, 0.9, .6, .5], 
         [.8, .2, .4, 0.9, .5, .4], [.8, .2, .4, 0.3, .2, .1]])
mu = [0.1, 0.2, 0.1, 0.15, 0.2, 0.18]
T = 100

### Find the optimal alpha using CPLEX

In [ ]:
def cp(M, mu, theta):

    n = len(mu)
    # --- MODEL ---
    mdl = Model(name='hawkes_adj_opt')
    
    # Decision vars
    alpha = {(i,j): mdl.binary_var(name=f"alpha_{i}_{j}")
             for i in range(n) for j in range(n)}
    #alpha = {(i,j): mdl.continuous_var(lb=0, ub=1, name=f"alpha_{i}_{j}") for i in range(n) for j in range(n)}
    
    g     = {(i,j): mdl.continuous_var(lb=0, name=f"g_{i}_{j}")
             for i in range(n) for j in range(n)}
    
    # 1) Diagonal ones
    for i in range(n):
        mdl.add_constraint(alpha[i,i] == 1, ctname=f"diag_{i}")
    
    # 2) Degree constraints
    for i in range(n):
        mdl.add_constraint(mdl.sum(alpha[i,j] for j in range(n)) == M+1,
                           ctname=f"outdeg_{i}")
    for j in range(n):
        mdl.add_constraint(mdl.sum(alpha[i,j] for i in range(n)) == M+1,
                           ctname=f"indeg_{j}")
    
    # 3) Indicator constraints for g_{ij} = alpha_{ij} * theta_j * λ̄_j
    #    where λ̄_j = mu[j] + sum_k g[j,k]
    for i in range(n):
        for j in range(n):
            # compute λ̄_j as a linear expr in the g-variables
            lam_bar_j = mu[j] + mdl.sum(g[j,k] for k in range(n))
            # if alpha=1, enforce g = theta_j * lambda_bar_j
            mdl.add_indicator(alpha[i,j], g[i,j] == theta[j] * lam_bar_j,
                              name=f"ind_on_{i}_{j}")
            # if alpha=0, enforce g=0
            mdl.add_indicator(alpha[i,j], g[i,j] == 0,
                             active_value=0, name=f"ind_off_{i}_{j}")
            #mdl.add_indicator(alpha[i,j].negate(), g[i,j] == 0,
            #                  name=f"ind_off_{i}_{j}")

    # 4) Symmetry: α[i,j] == α[j,i]
    for i in range(n):
        for j in range(i+1, n):
            mdl.add_constraint(alpha[i,j] == alpha[j,i],
                               ctname=f"sym_{i}_{j}")

    mdl.minimize(mu[i] + mdl.sum(g[i,j] for i in range(n) for j in range(n)))
    #mdl.minimize(mdl.sum(mu[i] for i in range(n)) + mdl.sum(g[i,j] for i in range(n) for j in range(n)))
    #mdl.minimize(mdl.sum(mu[i] for i in range(n)) + mdl.sum(g[i,j] for i in range(n) for j in range(n)))
    # Solve
    sol = mdl.solve()
    if sol is None:
        raise RuntimeError("No feasible solution found")

    g_sol = np.array([[sol.get_value(g[i,j])
                       for j in range(n)]
                      for i in range(n)])

    # Extract optimized alpha matrix
    alpha_opt = [[int(alpha[i,j].solution_value) for j in range(n)]
                 for i in range(n)]
    alpha_opt = np.array(alpha_opt)
    return alpha_opt
#M = 2
#Then we rescale theta to maintain keep the spectral radius of theta*alpha < 1
slack = 1e-6
target = (1 - slack) / (M)
scale  = target / max(theta)
theta_scaled = [t * scale for t in theta]
alpha_cplex = cp(M, mu, theta_scaled)
print(alpha_cplex)


### Prove that CPLEX gives the best alpha to minimize the subjective funtion

In [ ]:
def generate_2_regular_adjacency(n=6, include_diagonal=True):
    verts = list(range(n))
    out = []
    def backtrack(unused, edges):
        if not unused:
            A = np.zeros((n,n), dtype=int)
            for u,v in edges:
                A[u,v] = A[v,u] = 1
            if include_diagonal:
                np.fill_diagonal(A, 1)
            out.append(A)
            return
        start = min(unused)
        rest = sorted(unused - {start})
        for k in range(3, len(unused)+1):
            for subset in combinations(rest, k-1):
                for p in permutations(subset):
                    if p[0] >= p[-1]:
                        continue
                    cyc = [start] + list(p)
                    e2 = list(edges)
                    for i in range(k):
                        u,v = cyc[i], cyc[(i+1)%k]
                        e2.append((u,v) if u<v else (v,u))
                    backtrack(unused - set(cyc), e2)
    backtrack(unused=set(verts), edges=[])
    return out  # 70 for n=6

def subjective_function(alpha, mu, theta, require_stable=True):
    n = len(mu)
    B = alpha @ np.diag(theta)                   # nonnegative matrix
    if require_stable:
        # Perron–Frobenius spectral radius
        rho = max(abs(np.linalg.eigvals(B)))
        if rho >= 1 - 1e-12:
            return float('inf')

    M = np.eye(n) - B
    lam = np.linalg.solve(M, mu)
    
    return (alpha * theta* lam).sum()

M_degree = 2
#Rescale theta
slack = 1e-6

target = (1 - slack) / M_degree
scale  = target / float(np.max(theta))
theta_scaled = theta * scale

alphas = generate_2_regular_adjacency(n=len(mu), include_diagonal=True)
scores = np.array([subjective_function(A, mu, theta_scaled, require_stable=True) for A in alphas])

best_idx = int(np.argmin(scores))

best_alpha = alphas[best_idx]
print("Exhaustive best alpha:\n", best_alpha)
print('')
print(best_alpha - alpha_cplex)

